# Bug Prediction System — Model Training
**Input:** `szz_training_ready-2.csv`
- 2,176 real commits from vercel/next.js
- 115 bugs (5.28% bug rate)
- 3 time periods: 2018-2020, 2021-2023, 2024-2026
- Language: TypeScript

---

## Step 1 — Install & Import

In [ ]:
!pip install pandas numpy scikit-learn xgboost matplotlib seaborn joblib shap -q
print('✅ Done')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    roc_auc_score, classification_report, confusion_matrix,
    roc_curve, precision_score, recall_score, f1_score
)
from xgboost import XGBClassifier
import shap

print('✅ All imports done')

## Step 2 — Load Data

In [ ]:
df = pd.read_csv('szz_training_ready-2.csv')

print(f'Shape         : {df.shape}')
print(f'Bug rate      : {df["is_buggy"].mean():.1%}')
print(f'Total bugs    : {df["is_buggy"].sum()}')
print(f'Total clean   : {(df["is_buggy"]==0).sum()}')
print(f'\nTime periods:')
print(df['time_period'].value_counts())
print(f'\nBug rate by time period:')
print(df.groupby('time_period')['is_buggy'].mean().round(4))
print(f'\nMissing values: {df.isnull().sum().sum()} (should be 0)')

## Step 3 — Feature Engineering

In [ ]:
# Encode time_period (categorical → number)
le_period = LabelEncoder()
df['time_period_enc'] = le_period.fit_transform(df['time_period'])

print('Time period encoding:')
for i, c in enumerate(le_period.classes_):
    print(f'  {c} → {i}')

# NOTE: language_group dropped — all rows are TypeScript (zero variance)
# NOTE: confidence dropped — label metadata, not a real feature

FEATURE_COLS = [
    # Strongest predictors (from correlation analysis)
    'prior_bugs_author',       # 0.30 correlation — best feature
    'avg_complexity',          # complexity of changed methods
    'test_ratio',              # proportion of test files changed
    'test_files_changed',      # raw count of test files
    'complexity_per_file',     # complexity density
    # Moderate predictors
    'files_changed',
    'num_methods',
    'churn_ratio',
    'lines_added',
    'lines_deleted',
    # Timing (weak but included)
    'commit_hour',
    'day_of_week',
    'is_weekend',
    'is_night_commit',
    # Time period (research feature)
    'time_period_enc',
]

X = df[FEATURE_COLS].fillna(0)
y = df['is_buggy']

print(f'\n✅ Features : {X.shape[1]} columns')
print(f'   Rows     : {X.shape[0]}')
print(f'   Bugs     : {y.sum()} ({y.mean():.1%})')

## Step 4 — EDA Charts

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Bug Prediction — Data Overview (vercel/next.js | TypeScript)',
             fontsize=13, fontweight='bold')

# 1. Class balance
ax = axes[0, 0]
counts = y.value_counts().sort_index()
bars = ax.bar(['Clean', 'Buggy'], counts.values,
              color=['#2ecc71', '#e74c3c'], edgecolor='white', linewidth=1.5)
ax.set_title('Class Balance', fontweight='bold')
ax.set_ylabel('Count')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 10, str(val),
            ha='center', fontweight='bold', fontsize=11)

# 2. Bug rate by time period
ax = axes[0, 1]
tp = df.groupby('time_period')['is_buggy'].mean().sort_index()
bars2 = ax.bar(tp.index, tp.values,
               color=['#3498db', '#e67e22', '#9b59b6'], edgecolor='white')
ax.set_title('Bug Rate by Time Period', fontweight='bold')
ax.set_ylabel('Bug Rate')
ax.tick_params(axis='x', rotation=15)
for bar, val in zip(bars2, tp.values):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.001,
            f'{val:.1%}', ha='center', fontsize=10)

# 3. Correlation with is_buggy
ax = axes[0, 2]
corr = X.corrwith(y).sort_values(ascending=False)
colors = ['#e74c3c' if v > 0 else '#3498db' for v in corr.values]
ax.barh(corr.index, corr.values, color=colors, edgecolor='white')
ax.set_title('Feature Correlation with is_buggy', fontweight='bold')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Correlation')

# 4. Lines added (clipped)
ax = axes[1, 0]
clean_l = df[df['is_buggy']==0]['lines_added'].clip(0, 500)
buggy_l = df[df['is_buggy']==1]['lines_added'].clip(0, 500)
ax.hist(clean_l, bins=30, alpha=0.6, color='#2ecc71', label='Clean')
ax.hist(buggy_l, bins=30, alpha=0.7, color='#e74c3c', label='Buggy')
ax.set_title('Lines Added Distribution (clipped 500)', fontweight='bold')
ax.set_xlabel('Lines Added')
ax.legend()

# 5. Prior bugs author
ax = axes[1, 1]
prior = df.groupby('prior_bugs_author')['is_buggy'].mean().head(10)
ax.bar(prior.index.astype(str), prior.values,
       color='#9b59b6', edgecolor='white')
ax.set_title('Bug Rate by Prior Bugs (Author)', fontweight='bold')
ax.set_xlabel('Prior bugs this author caused')
ax.set_ylabel('Bug Rate')

# 6. Commits per time period
ax = axes[1, 2]
tp_count = df.groupby('time_period').size().sort_index()
ax.bar(tp_count.index, tp_count.values,
       color='#1abc9c', edgecolor='white')
ax.set_title('Commits per Time Period', fontweight='bold')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=15)
for i, (idx, val) in enumerate(tp_count.items()):
    ax.text(i, val + 5, str(val), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA saved')

## Step 5 — Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y    # keeps 5% bug rate in both sets
)

print(f'Training set  : {X_train.shape[0]} commits')
print(f'  Bugs        : {y_train.sum()} ({y_train.mean():.1%})')
print(f'  Clean       : {(y_train==0).sum()}')
print(f'\nTest set      : {X_test.shape[0]} commits')
print(f'  Bugs        : {y_test.sum()} ({y_test.mean():.1%})')
print(f'  Clean       : {(y_test==0).sum()}')

## Step 6 — Train Random Forest

In [ ]:
rf_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('clf',     RandomForestClassifier(
                    n_estimators=200,
                    class_weight='balanced',
                    max_depth=10,
                    min_samples_leaf=5,
                    random_state=42,
                    n_jobs=-1
                ))
])

rf_model.fit(X_train, y_train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf_cv = cross_val_score(rf_model, X_train, y_train,
                        cv=cv, scoring='roc_auc')

print(f'✅ Random Forest trained')
print(f'   CV AUC scores : {rf_cv.round(3)}')
print(f'   CV AUC mean   : {rf_cv.mean():.3f} ± {rf_cv.std():.3f}')

## Step 7 — Train XGBoost

In [ ]:
scale = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight = {scale:.1f}')

xgb_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('clf',     XGBClassifier(
                    n_estimators=200,
                    max_depth=5,
                    learning_rate=0.05,
                    scale_pos_weight=scale,
                    random_state=42,
                    eval_metric='auc',
                    verbosity=0
                ))
])

xgb_model.fit(X_train, y_train)

xgb_cv = cross_val_score(xgb_model, X_train, y_train,
                         cv=cv, scoring='roc_auc')

print(f'✅ XGBoost trained')
print(f'   CV AUC scores : {xgb_cv.round(3)}')
print(f'   CV AUC mean   : {xgb_cv.mean():.3f} ± {xgb_cv.std():.3f}')

## Step 8 — Pick Best Model & Save

In [ ]:
print('=== MODEL COMPARISON ===')
print(f'Random Forest CV AUC : {rf_cv.mean():.3f}')
print(f'XGBoost       CV AUC : {xgb_cv.mean():.3f}')

if rf_cv.mean() >= xgb_cv.mean():
    best_model = rf_model
    best_name  = 'Random Forest'
else:
    best_model = xgb_model
    best_name  = 'XGBoost'

print(f'\n✅ Winner : {best_name}')

joblib.dump(best_model,  'bug_prediction_model.pkl')
joblib.dump(le_period,   'label_encoder_period.pkl')
joblib.dump(FEATURE_COLS,'feature_cols.pkl')
print('   Saved  : bug_prediction_model.pkl')

## Step 9 — Evaluate on Test Set

In [ ]:
y_proba = best_model.predict_proba(X_test)[:, 1]
y_pred  = best_model.predict(X_test)

auc       = roc_auc_score(y_test, y_proba)
precision = precision_score(y_test, y_pred, zero_division=0)
recall    = recall_score(y_test, y_pred, zero_division=0)
f1        = f1_score(y_test, y_pred, zero_division=0)

print('=' * 50)
print(f'  TEST RESULTS — {best_name}')
print('=' * 50)
print(f'  AUC-ROC   : {auc:.3f}')
print(f'  Precision : {precision:.3f}')
print(f'  Recall    : {recall:.3f}')
print(f'  F1 Score  : {f1:.3f}')
print('=' * 50)
print()
print(classification_report(y_test, y_pred,
      target_names=['Clean', 'Buggy'], zero_division=0))

## Step 10 — Evaluation Charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'Model Evaluation — {best_name}', fontsize=14, fontweight='bold')

# ROC Curve
ax = axes[0]
fpr, tpr, _ = roc_curve(y_test, y_proba)
ax.plot(fpr, tpr, color='#e74c3c', linewidth=2.5,
        label=f'AUC = {auc:.3f}')
ax.plot([0,1],[0,1],'k--', alpha=0.4, label='Random (0.5)')
ax.fill_between(fpr, tpr, alpha=0.1, color='#e74c3c')
ax.set_title('ROC Curve', fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend()

# Confusion Matrix
ax = axes[1]
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', ax=ax,
            xticklabels=['Pred Clean','Pred Buggy'],
            yticklabels=['Actual Clean','Actual Buggy'],
            linewidths=0.5)
ax.set_title('Confusion Matrix', fontweight='bold')

# Feature Importance
ax = axes[2]
clf = best_model.named_steps['clf']
if hasattr(clf, 'feature_importances_'):
    imp = clf.feature_importances_
    idx = np.argsort(imp)
    ax.barh([FEATURE_COLS[i] for i in idx],
            imp[idx], color='#3498db', edgecolor='white')
    ax.set_title('Feature Importance', fontweight='bold')
    ax.set_xlabel('Importance')

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Charts saved to model_evaluation.png')

## Step 11 — Time Period Analysis
Bug prediction accuracy across different time periods

In [ ]:
# Add predictions back to full dataframe
df_full = df.copy()
X_full = df_full[FEATURE_COLS].fillna(0)
df_full['predicted_prob'] = best_model.predict_proba(X_full)[:, 1]
df_full['predicted_bug']  = best_model.predict(X_full)

print('=== TIME PERIOD ANALYSIS ===')
for period in sorted(df_full['time_period'].unique()):
    subset = df_full[df_full['time_period'] == period]
    actual_rate    = subset['is_buggy'].mean()
    predicted_rate = subset['predicted_prob'].mean()
    count          = len(subset)
    bugs           = subset['is_buggy'].sum()

    # AUC per period if enough bugs
    if bugs >= 5:
        period_auc = roc_auc_score(
            subset['is_buggy'],
            subset['predicted_prob']
        )
        auc_str = f'{period_auc:.3f}'
    else:
        auc_str = 'n/a (too few bugs)'

    print(f'\n  Period : {period}')
    print(f'  Commits: {count}  |  Bugs: {bugs}')
    print(f'  Actual bug rate    : {actual_rate:.1%}')
    print(f'  Predicted avg risk : {predicted_rate:.1%}')
    print(f'  AUC                : {auc_str}')

# Chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Time Period Analysis', fontsize=13, fontweight='bold')

tp_actual    = df_full.groupby('time_period')['is_buggy'].mean().sort_index()
tp_predicted = df_full.groupby('time_period')['predicted_prob'].mean().sort_index()

x = np.arange(len(tp_actual))
axes[0].bar(x - 0.2, tp_actual.values,   0.4,
            label='Actual',    color='#e74c3c', edgecolor='white')
axes[0].bar(x + 0.2, tp_predicted.values, 0.4,
            label='Predicted', color='#3498db', edgecolor='white')
axes[0].set_xticks(x)
axes[0].set_xticklabels(tp_actual.index, rotation=15)
axes[0].set_title('Actual vs Predicted Bug Rate by Era', fontweight='bold')
axes[0].set_ylabel('Bug Rate')
axes[0].legend()

tp_count = df_full.groupby('time_period').size().sort_index()
axes[1].bar(tp_count.index, tp_count.values,
            color='#1abc9c', edgecolor='white')
axes[1].set_title('Commits per Time Period', fontweight='bold')
axes[1].set_ylabel('Commits')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('time_period_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Time period chart saved')

## Step 12 — SHAP Explanations

In [ ]:
X_test_proc = best_model[:-1].transform(X_test)
clf         = best_model.named_steps['clf']

explainer   = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_test_proc)

sv = shap_values[1] if isinstance(shap_values, list) else shap_values

plt.figure(figsize=(10, 6))
shap.summary_plot(sv, X_test_proc,
                  feature_names=FEATURE_COLS,
                  show=False, plot_type='bar')
plt.title('SHAP — What drives bug predictions?', fontweight='bold')
plt.tight_layout()
plt.savefig('shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ SHAP chart saved')

## Step 13 — Predict Any New Commit

In [ ]:
def predict_commit(commit_dict):
    # Encode time_period if provided
    period = commit_dict.get('time_period', '2024-2026')
    if period in le_period.classes_:
        commit_dict['time_period_enc'] = le_period.transform([period])[0]
    else:
        commit_dict['time_period_enc'] = 2  # default to latest

    row  = pd.DataFrame([{col: commit_dict.get(col, 0) for col in FEATURE_COLS}])
    prob = best_model.predict_proba(row)[0][1]

    risk = ('🔴 HIGH RISK'   if prob >= 0.6 else
            '🟡 MEDIUM RISK' if prob >= 0.3 else
            '🟢 LOW RISK')

    print(f'Bug probability : {prob:.1%}')
    print(f'Risk level      : {risk}')
    return prob, risk


# Example 1 — Risky commit
print('=== Example 1: Risky Commit ===')
predict_commit({
    'lines_added'        : 800,
    'lines_deleted'      : 200,
    'files_changed'      : 25,
    'avg_complexity'     : 15.0,
    'num_methods'        : 30,
    'test_files_changed' : 0,
    'test_ratio'         : 0.0,
    'complexity_per_file': 0.6,
    'churn_ratio'        : 4.0,
    'prior_bugs_author'  : 8,
    'commit_hour'        : 23,
    'day_of_week'        : 6,
    'is_weekend'         : 1,
    'is_night_commit'    : 1,
    'time_period'        : '2024-2026',
})

print()

# Example 2 — Safe commit
print('=== Example 2: Safe Commit ===')
predict_commit({
    'lines_added'        : 12,
    'lines_deleted'      : 3,
    'files_changed'      : 2,
    'avg_complexity'     : 1.5,
    'num_methods'        : 2,
    'test_files_changed' : 2,
    'test_ratio'         : 1.0,
    'complexity_per_file': 0.75,
    'churn_ratio'        : 4.0,
    'prior_bugs_author'  : 0,
    'commit_hour'        : 10,
    'day_of_week'        : 1,
    'is_weekend'         : 0,
    'is_night_commit'    : 0,
    'time_period'        : '2024-2026',
})

## Step 14 — Final Summary

In [ ]:
clf = best_model.named_steps['clf']

print('=' * 55)
print('  BUG PREDICTION MODEL — FINAL SUMMARY')
print('=' * 55)
print(f'  Repo            : vercel/next.js (TypeScript)')
print(f'  Time periods    : 2018-2020, 2021-2023, 2024-2026')
print(f'  Total commits   : {len(df):,}')
print(f'  Bug count       : {df["is_buggy"].sum()} ({df["is_buggy"].mean():.1%})')
print(f'  Features used   : {len(FEATURE_COLS)}')
print(f'  Best model      : {best_name}')
print(f'  Test AUC        : {auc:.3f}')
print(f'  Precision       : {precision:.3f}')
print(f'  Recall          : {recall:.3f}')
print(f'  F1 Score        : {f1:.3f}')
print('=' * 55)

if hasattr(clf, 'feature_importances_'):
    top3 = np.argsort(clf.feature_importances_)[-3:][::-1]
    print(f'\n  Top 3 Predictors:')
    for i, idx in enumerate(top3):
        print(f'    {i+1}. {FEATURE_COLS[idx]} '
              f'(importance={clf.feature_importances_[idx]:.3f})')

print(f'\n  Saved files:')
print(f'    bug_prediction_model.pkl')
print(f'    label_encoder_period.pkl')
print(f'    feature_cols.pkl')
print(f'    eda_overview.png')
print(f'    model_evaluation.png')
print(f'    time_period_analysis.png')
print(f'    shap_importance.png')
print('=' * 55)